# WoundDoc Lite — Full Training Pipeline
## Hackathon Edition: Open-Source Datasets → Fine-tuned Wound Concern Model → Structured Output

---

### What this notebook does

| Stage | What happens |
|---|---|
| **0** | Install all dependencies |
| **1** | Download open-source wound datasets (Kaggle + GitHub) |
| **2** | Build a unified label map: `pressure_injury / other_wound / intact_skin` |
| **3** | Compute colour, texture and morphology features for every ROI |
| **4** | Fine-tune a lightweight MobileNetV3 binary concern head (accessible, edge-deployable) |
| **5** | Load CLIP zero-shot fallback for uncertain predictions |
| **6** | RAG layer: embed NPUAP/EPUAP pressure-injury guideline chunks into FAISS |
| **7** | Full inference pipeline: image → ROI → concern score → retrieval → JSON note |
| **8** | Evaluation: AUROC, sensitivity, specificity, abstention rate |
| **9** | Save and package all artefacts for the demo notebook |

### Important
> This is **for demo / hackathon purposes only**.  
> Outputs are **not** clinically validated and must **never** replace clinician judgment.

### Datasets used (all open-source / freely accessible)
- **Kaggle Wound Segmentation** — `leoscode/wound-segmentation-images` (~600 images, requires kaggle.json)
- **UWM Wound Segmentation** — GitHub `uwm-bigdata/wound-segmentation` (~350 images, no auth needed)

### Pretrained backbones
- **MobileNetV3-Small** (torchvision, ~2.5 MB) — lightweight, edge-ready, fast fine-tune
- **CLIP ViT-B/32** (open_clip, laion2b) — zero-shot fallback + semantic embedding
- **all-MiniLM-L6-v2** (sentence-transformers, ~22 MB) — RAG guideline embeddings


---
## Stage 0 — Install Dependencies

In [ ]:
# =============================================================
# Stage 0 — Dependencies
# =============================================================
%pip -q install \
    opencv-python-headless \
    open_clip_torch \
    ftfy \
    kaggle \
    scikit-learn \
    scikit-image \
    matplotlib \
    tqdm \
    sentence-transformers \
    faiss-cpu \
    albumentations \
    torchmetrics \
    gdown

print("All dependencies installed.")

---
## Stage 1 — Imports, Config & Seeds

In [ ]:
# =============================================================
# Stage 1 — Imports & Global Config
# =============================================================
import os, sys, json, shutil, zipfile, random, warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms

import albumentations as A
from albumentations.pytorch import ToTensorV2

import open_clip
from sklearn.metrics import (
    roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split
from skimage.feature import local_binary_pattern

from sentence_transformers import SentenceTransformer
import faiss

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

# ── Device ───────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── Directory layout ─────────────────────────────────────────
BASE_DIR    = Path("/content/wounddoc_train")
DATA_DIR    = BASE_DIR / "data"
UNIFIED_DIR = BASE_DIR / "unified"
MODEL_DIR   = BASE_DIR / "models"
RAG_DIR     = BASE_DIR / "rag"
OUTPUT_DIR  = BASE_DIR / "outputs"

for d in [DATA_DIR, UNIFIED_DIR, MODEL_DIR, RAG_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Label map ────────────────────────────────────────────────
# 0 = pressure_injury  (positive class — what we care about)
# 1 = other_wound      (other chronic wounds)
# 2 = intact_skin      (healthy / uncertain)
LABEL_MAP   = {"pressure_injury": 0, "other_wound": 1, "intact_skin": 2}
LABEL_NAMES = ["pressure_injury", "other_wound", "intact_skin"]

print("Config ready. Label map:", LABEL_MAP)

---
## Stage 2 — Dataset Download & Unification

| Source | Images | Labels |
|---|---|---|
| Kaggle `leoscode/wound-segmentation-images` | ~600 | wound masks (treated as other_wound) |
| UWM GitHub wound_dataset | ~350 | pressure / diabetic / venous / surgical folder names |

> Upload your `kaggle.json` to `/content/` before running Cell 2a.  
> The pipeline works with UWM alone if Kaggle is unavailable.

In [ ]:
# =============================================================
# Stage 2a — Kaggle API setup (optional)
# =============================================================
KAGGLE_JSON = Path("/content/kaggle.json")
USE_KAGGLE  = KAGGLE_JSON.exists()

if USE_KAGGLE:
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    shutil.copy(KAGGLE_JSON, os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("Kaggle credentials configured.")
else:
    print("kaggle.json not found. Using UWM dataset only.")
    print("To add: upload kaggle.json to /content/ and re-run.")

In [ ]:
# =============================================================
# Stage 2b — Download Kaggle wound segmentation dataset
# =============================================================
KAGGLE_DIR = DATA_DIR / "kaggle_wound"
KAGGLE_DIR.mkdir(exist_ok=True)

if USE_KAGGLE and not any(KAGGLE_DIR.iterdir()) if KAGGLE_DIR.exists() else True:
    !kaggle datasets download -d leoscode/wound-segmentation-images -p "{KAGGLE_DIR}" --unzip -q
    print("Kaggle dataset downloaded.")
elif USE_KAGGLE:
    print("Kaggle dataset already present.")
else:
    print("Skipping Kaggle download.")

In [ ]:
# =============================================================
# Stage 2c — Download UWM GitHub wound dataset (no auth needed)
# https://github.com/uwm-bigdata/wound-segmentation
# =============================================================
import urllib.request

UWM_DIR     = DATA_DIR / "uwm_wound"
UWM_DIR.mkdir(exist_ok=True)
UWM_ARCHIVE = UWM_DIR / "wound_dataset.zip"

if not UWM_ARCHIVE.exists():
    print("Downloading UWM wound dataset from GitHub...")
    uwm_url = "https://github.com/uwm-bigdata/wound-segmentation/archive/refs/heads/master.zip"
    urllib.request.urlretrieve(uwm_url, UWM_ARCHIVE)
    print("Downloaded:", UWM_ARCHIVE)

UWM_EXTRACTED = UWM_DIR / "extracted"
if not UWM_EXTRACTED.exists():
    with zipfile.ZipFile(UWM_ARCHIVE, 'r') as zf:
        zf.extractall(UWM_EXTRACTED)
    print("Extracted to", UWM_EXTRACTED)
else:
    print("UWM already extracted.")

# Show directory structure
for root, dirs, files in os.walk(UWM_EXTRACTED):
    lvl = root.replace(str(UWM_EXTRACTED), '').count(os.sep)
    if lvl > 3: break
    print('  ' * lvl + os.path.basename(root) + f'/  [{len(files)} files]')

In [ ]:
# =============================================================
# Stage 2d — Build unified manifest with pressure-injury labels
# UWM folder names encode wound type:
#   Pressure_Wound*    → 0 (pressure_injury)
#   Diabetic/Venous/Surgical → 1 (other_wound)
#   Normal/Healthy     → 2 (intact_skin)
# Kaggle: no PI labels → treat all as 1 (other_wound negatives)
# =============================================================
UWM_LABEL_MAP = {
    "pressure": 0, "diabetic": 1, "venous": 1,
    "surgical": 1, "normal":   2, "healthy": 2, "intact": 2,
}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
records  = []

# ── UWM ──────────────────────────────────────────────────────
for img_path in UWM_EXTRACTED.rglob("*"):
    if img_path.suffix.lower() not in IMG_EXTS:
        continue
    label = None
    for folder in [img_path.parent.name, img_path.parent.parent.name]:
        for key, lbl in UWM_LABEL_MAP.items():
            if key in folder.lower():
                label = lbl
                break
        if label is not None:
            break
    if label is None:
        label = 1  # default to other_wound
    records.append({"image_path": str(img_path), "label": label,
                    "label_name": LABEL_NAMES[label], "source": "uwm"})

# ── Kaggle (if downloaded) ────────────────────────────────────
if USE_KAGGLE and KAGGLE_DIR.exists():
    for img_path in KAGGLE_DIR.rglob("*"):
        if img_path.suffix.lower() not in IMG_EXTS:
            continue
        records.append({"image_path": str(img_path), "label": 1,
                        "label_name": "other_wound", "source": "kaggle"})

manifest = pd.DataFrame(records).drop_duplicates("image_path")
manifest = manifest[manifest["image_path"].apply(os.path.exists)].reset_index(drop=True)

print(f"Total usable images: {len(manifest)}")
print(manifest["label_name"].value_counts())

manifest.to_csv(UNIFIED_DIR / "manifest.csv", index=False)
print("Manifest saved.")

---
## Stage 3 — Wound Feature Extraction

Classical computer-vision features that power the structured note output:

- **Contour metrics**: area (px + mm²), perimeter/circumference, bbox, circularity
- **Tissue colour classification**: granulation / slough / eschar / epithelialising (HSV-based)
- **LBP texture**: 64-bin Local Binary Pattern histogram
- **GrabCut ROI**: consistent wound localisation

In [ ]:
# =============================================================
# Stage 3a — Feature extraction functions
# =============================================================

def compute_quality_flags(rgb: np.ndarray) -> List[str]:
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    flags = []
    if cv2.Laplacian(gray, cv2.CV_64F).var() < 50:  flags.append("blurry")
    b = gray.mean()
    if b < 50:    flags.append("low_light")
    elif b > 220: flags.append("overexposed")
    if gray.std() < 25: flags.append("low_contrast")
    return flags


def grabcut_roi(rgb: np.ndarray, rect_scale: float = 0.70, iters: int = 5) -> np.ndarray:
    h, w = rgb.shape[:2]
    x  = int((1 - rect_scale) * w / 2)
    y  = int((1 - rect_scale) * h / 2)
    rw = int(w * rect_scale)
    rh = int(h * rect_scale)
    gc_mask = np.zeros((h, w), np.uint8)
    bgd = np.zeros((1, 65), np.float64)
    fgd = np.zeros((1, 65), np.float64)
    try:
        cv2.grabCut(rgb, gc_mask, (x, y, rw, rh), bgd, fgd, iters, cv2.GC_INIT_WITH_RECT)
        mask = np.where((gc_mask == 2) | (gc_mask == 0), 0, 1).astype(np.uint8)
    except Exception:
        mask = np.zeros((h, w), np.uint8)
    k = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    return mask


def largest_component(mask: np.ndarray) -> np.ndarray:
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    if n <= 1: return mask.astype(np.uint8)
    idx = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    out = np.zeros_like(mask, np.uint8)
    out[labels == idx] = 1
    return out


def contour_metrics(mask: np.ndarray, px_per_mm: float = 1.0) -> Dict:
    m8 = (mask * 255).astype(np.uint8)
    contours, _ = cv2.findContours(m8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return {"area_px": 0, "perimeter_px": 0, "circularity": 0,
                "bbox": None, "area_mm2": 0, "perimeter_mm": 0}
    c     = max(contours, key=cv2.contourArea)
    area  = float(cv2.contourArea(c))
    perim = float(cv2.arcLength(c, True))
    x, y, bw, bh = cv2.boundingRect(c)
    circ  = (4 * np.pi * area / (perim ** 2)) if perim > 0 else 0.0
    return {
        "area_px":      area,
        "perimeter_px": perim,
        "circularity":  round(circ, 4),
        "bbox":         [int(x), int(y), int(x+bw), int(y+bh)],
        "area_mm2":     round(area / (px_per_mm ** 2), 2),
        "perimeter_mm": round(perim / px_per_mm, 2),
    }


TISSUE_COLOURS = {
    "granulation":     ([0,  50,  50], [20, 255, 255]),
    "slough":          ([15, 30, 100], [35, 200, 255]),
    "eschar":          ([0,   0,   0], [180, 80,  60]),
    "epithelialising": ([0,  10, 150], [20,  80, 255]),
}

def tissue_colour_ratios(rgb: np.ndarray, mask: np.ndarray) -> Dict:
    if mask.sum() == 0:
        return {t: 0.0 for t in TISSUE_COLOURS}
    hsv   = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    wound = hsv[mask > 0]
    total = len(wound)
    return {
        t: round(float(np.all((wound >= np.array(lo, np.uint8)) &
                              (wound <= np.array(hi, np.uint8)), axis=1).sum()) / total, 4)
        for t, (lo, hi) in TISSUE_COLOURS.items()
    }


def lbp_histogram(gray: np.ndarray, mask: np.ndarray,
                  n_points: int = 24, radius: int = 3, n_bins: int = 64) -> np.ndarray:
    lbp = local_binary_pattern(gray, n_points, radius, method="uniform")
    roi_vals = lbp[mask > 0] if mask.sum() > 0 else lbp.ravel()
    hist, _  = np.histogram(roi_vals, bins=n_bins, range=(0, n_points + 2), density=True)
    return hist.astype(np.float32)


def extract_wound_features(img_path: str) -> Optional[Dict]:
    bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if bgr is None: return None
    rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    mask = largest_component(grabcut_roi(rgb))
    ys, xs = np.where(mask > 0)
    bbox = [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())] if len(xs) > 0 else None
    return {
        "image_path":    img_path,
        "roi_found":     bbox is not None,
        "bbox":          bbox,
        **contour_metrics(mask),
        "tissue_colours": tissue_colour_ratios(rgb, mask),
        "lbp_histogram":  lbp_histogram(gray, mask).tolist(),
        "mean_lab":       (cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)[mask > 0].mean(axis=0).tolist()
                           if mask.sum() > 0 else [0.0, 0.0, 0.0]),
        "quality_flags":  compute_quality_flags(rgb),
    }

print("Feature extraction functions ready.")

In [ ]:
# =============================================================
# Stage 3b — Compute and cache features for all images
# =============================================================
FEATURE_CACHE = UNIFIED_DIR / "features.jsonl"

if not FEATURE_CACHE.exists():
    print(f"Extracting features for {len(manifest)} images...")
    with open(FEATURE_CACHE, "w") as fout:
        for _, row in tqdm(manifest.iterrows(), total=len(manifest)):
            feat = extract_wound_features(row["image_path"])
            if feat is None: continue
            feat["label"]      = int(row["label"])
            feat["label_name"] = row["label_name"]
            feat["source"]     = row["source"]
            fout.write(json.dumps(feat) + "\n")
    print("Features saved.", FEATURE_CACHE)
else:
    print("Feature cache already exists.")

feat_df = pd.read_json(FEATURE_CACHE, lines=True)
print(f"Feature records: {len(feat_df)}")
print(feat_df["label_name"].value_counts())

---
## Stage 4 — Fine-tune MobileNetV3-Small Wound Concern Classifier

Training strategy:
1. **Phase 1 (frozen)** — 4 epochs, head-only training
2. **Phase 2 (partial unfreeze)** — 4 more epochs, last 3 blocks + head
3. **Augmentation** — flip, rotate, colour jitter, elastic, noise
4. **Weighted sampler** — handles class imbalance

In [ ]:
# =============================================================
# Stage 4a — PyTorch Dataset + augmentation
# =============================================================
IMG_SIZE = 224

TRAIN_TRANSFORMS = A.Compose([
    A.RandomResizedCrop(IMG_SIZE, IMG_SIZE, scale=(0.7, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=30, p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05, p=0.6),
    A.GaussNoise(var_limit=(10, 50), p=0.3),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.2),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

VAL_TRANSFORMS = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])


class WoundDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row["image_path"], cv2.IMREAD_COLOR)
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, int(row["label"])


# ── Stratified splits ─────────────────────────────────────────
valid_paths  = set(feat_df["image_path"].tolist())
use_manifest = manifest[manifest["image_path"].isin(valid_paths)].copy()

train_df, test_df = train_test_split(
    use_manifest, test_size=0.15, random_state=SEED, stratify=use_manifest["label"]
)
train_df, val_df  = train_test_split(
    train_df, test_size=0.15, random_state=SEED, stratify=train_df["label"]
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("Train label distribution:")
print(train_df["label_name"].value_counts())

# ── Weighted sampler ──────────────────────────────────────────
label_counts   = train_df["label"].value_counts().sort_index()
class_weights  = 1.0 / label_counts.values.astype(float)
sample_weights = np.array([class_weights[lbl] for lbl in train_df["label"].values])
sampler        = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

BATCH = 32
train_loader = DataLoader(WoundDataset(train_df, TRAIN_TRANSFORMS), batch_size=BATCH, sampler=sampler,  num_workers=2)
val_loader   = DataLoader(WoundDataset(val_df,   VAL_TRANSFORMS),   batch_size=BATCH, shuffle=False,     num_workers=2)
test_loader  = DataLoader(WoundDataset(test_df,  VAL_TRANSFORMS),   batch_size=BATCH, shuffle=False,     num_workers=2)

print(f"Loaders ready. Batch={BATCH}")

In [ ]:
# =============================================================
# Stage 4b — MobileNetV3-Small model definition
# =============================================================
NUM_CLASSES = len(LABEL_MAP)  # 3


def build_mobilenet(num_classes: int = 3, dropout: float = 0.3) -> nn.Module:
    backbone = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    in_features = backbone.classifier[0].in_features
    backbone.classifier = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.Hardswish(),
        nn.Dropout(p=dropout),
        nn.Linear(256, num_classes),
    )
    return backbone


def freeze_backbone(model: nn.Module) -> nn.Module:
    for name, param in model.named_parameters():
        param.requires_grad = ("classifier" in name)
    return model


def partial_unfreeze(model: nn.Module) -> nn.Module:
    total_blocks = len(list(model.features.children()))
    for name, param in model.named_parameters():
        if "features" in name:
            parts = name.split(".")
            block_idx = int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else -1
            param.requires_grad = (block_idx >= total_blocks - 3)
        else:
            param.requires_grad = True
    return model


model = freeze_backbone(build_mobilenet(NUM_CLASSES)).to(DEVICE)
n_tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_tot = sum(p.numel() for p in model.parameters())
print(f"MobileNetV3-Small | Trainable: {n_tr:,} / {n_tot:,} params")

In [ ]:
# =============================================================
# Stage 4c — Training loop (head-only → partial unfreeze)
# =============================================================
EPOCHS_FROZEN   = 4
EPOCHS_UNFREEZE = 4
LR_FROZEN       = 1e-3
LR_UNFREEZE     = 3e-4


def run_epoch(model, loader, criterion, optimizer=None, phase="train"):
    is_train = (phase == "train")
    model.train(is_train)
    total_loss = correct = total = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            probs = F.softmax(logits, dim=1).detach().cpu().numpy()
            preds = logits.argmax(dim=1).detach().cpu().numpy()
            lbls  = labels.detach().cpu().numpy()
            total_loss += loss.item() * len(lbls)
            correct    += (preds == lbls).sum()
            total      += len(lbls)
            all_preds.extend(preds.tolist())
            all_labels.extend(lbls.tolist())
            all_probs.extend(probs.tolist())

    return total_loss/total, correct/total, all_preds, all_labels, all_probs


def train_phase(model, epochs, lr, tag):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history, best_vloss, best_state = [], float("inf"), None

    for ep in range(1, epochs+1):
        tl, ta, *_ = run_epoch(model, train_loader, criterion, optimizer, "train")
        vl, va, *_ = run_epoch(model, val_loader,   criterion, phase="val")
        scheduler.step()
        history.append({"phase": tag, "epoch": ep, "tr_loss": tl, "tr_acc": ta,
                        "vl_loss": vl, "vl_acc": va})
        if vl < best_vloss:
            best_vloss = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"[{tag}] Ep {ep:02d}/{epochs} "
              f"tr_loss={tl:.4f} tr_acc={ta:.3f} "
              f"vl_loss={vl:.4f} vl_acc={va:.3f}")

    if best_state: model.load_state_dict(best_state)
    return model, history


# Phase 1
print("=== Phase 1: Head-only ===")
model, h1 = train_phase(model, EPOCHS_FROZEN, LR_FROZEN, "frozen")

# Phase 2
print("\n=== Phase 2: Partial unfreeze ===")
model = partial_unfreeze(model)
print(f"Trainable after unfreeze: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
model, h2 = train_phase(model, EPOCHS_UNFREEZE, LR_UNFREEZE, "unfreeze")

history = h1 + h2

# Save
MODEL_PATH = MODEL_DIR / "wound_concern_mobilenet.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "label_map":        LABEL_MAP,
    "label_names":      LABEL_NAMES,
    "img_size":         IMG_SIZE,
    "architecture":     "mobilenet_v3_small",
}, MODEL_PATH)
print(f"\nModel saved → {MODEL_PATH}")

In [ ]:
# =============================================================
# Stage 4d — Training curves
# =============================================================
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (tr_col, vl_col), title in zip(
    axes, [("tr_loss","vl_loss"),("tr_acc","vl_acc")], ["Loss","Accuracy"]
):
    ax.plot(hist_df[tr_col].values, label="Train",      marker="o")
    ax.plot(hist_df[vl_col].values, label="Validation", marker="s")
    ax.axvline(EPOCHS_FROZEN - 0.5, color="gray", ls="--", label="Unfreeze")
    ax.set_xlabel("Epoch"); ax.set_title(f"Training {title}")
    ax.legend(); ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print("Saved training_curves.png")

---
## Stage 5 — Evaluation

In [ ]:
# =============================================================
# Stage 5a — Test set evaluation
# =============================================================
crit = nn.CrossEntropyLoss()
_, test_acc, test_preds, test_labels, test_probs = run_epoch(model, test_loader, crit, phase="val")
test_probs_np = np.array(test_probs)

print("\n=== Test Set Results ===")
print(f"Accuracy: {test_acc:.4f}")
print()
print(classification_report(test_labels, test_preds, target_names=LABEL_NAMES, digits=4))

# AUROC: pressure_injury class (one-vs-rest)
pi_true   = [1 if l == 0 else 0 for l in test_labels]
pi_scores = test_probs_np[:, 0]
try:
    auroc = roc_auc_score(pi_true, pi_scores)
    print(f"Pressure-Injury AUROC (OvR): {auroc:.4f}")
except Exception as e:
    auroc = None
    print(f"AUROC could not be computed: {e}")

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=LABEL_NAMES).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Test Confusion Matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show(); plt.close()

In [ ]:
# =============================================================
# Stage 5b — Abstention / uncertainty analysis
# If max softmax < ABSTAIN_THRESHOLD → output "unable to determine"
# =============================================================
ABSTAIN_THRESHOLD = 0.50

max_probs    = test_probs_np.max(axis=1)
abstained    = max_probs < ABSTAIN_THRESHOLD
abstain_rate = abstained.mean()
preds_arr    = np.array(test_preds)
labels_arr   = np.array(test_labels)
conf_acc     = (preds_arr[~abstained] == labels_arr[~abstained]).mean() if (~abstained).sum() > 0 else 0.0

print(f"Abstention threshold:    {ABSTAIN_THRESHOLD}")
print(f"Abstention rate:         {abstain_rate:.3f}  ({abstained.sum()}/{len(abstained)})")
print(f"Accuracy (non-abstained):{conf_acc:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(max_probs, bins=40, edgecolor="black", alpha=0.75)
ax.axvline(ABSTAIN_THRESHOLD, color="red", ls="--", label=f"Abstain={ABSTAIN_THRESHOLD}")
ax.set_xlabel("Max softmax probability"); ax.set_ylabel("Count")
ax.set_title("Confidence distribution (test set)"); ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.show(); plt.close()

---
## Stage 6 — CLIP Zero-Shot Fallback

When MobileNet abstains, CLIP fills the gap using clinically grounded
multi-prompt averaging per class.
This ensures the system never silently fails.

In [ ]:
# =============================================================
# Stage 6 — CLIP zero-shot fallback
# =============================================================
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k", device=DEVICE
)
clip_model.eval()
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")

# Multi-prompt ensemble per class (mean-pooled)
CLIP_PROMPTS = {
    "pressure_injury": [
        "a clinical photograph of a pressure injury on a patient",
        "a bedsore or pressure ulcer on skin over a bony prominence",
        "a stage 2 or stage 3 pressure wound with erythema or tissue loss",
    ],
    "other_wound": [
        "a clinical photograph of a diabetic foot ulcer",
        "a venous leg ulcer with slough and irregular margins",
        "a surgical incision or post-operative wound",
    ],
    "intact_skin": [
        "a clinical photograph of healthy intact skin without any wound",
        "normal skin without a wound or lesion",
    ],
}

class_text_feats = {}
for cls, prompts in CLIP_PROMPTS.items():
    tokens = clip_tokenizer(prompts).to(DEVICE)
    with torch.no_grad():
        feats = clip_model.encode_text(tokens)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        class_text_feats[cls] = feats.mean(dim=0, keepdim=True)

text_matrix = torch.cat([class_text_feats[c] for c in LABEL_NAMES], dim=0)
text_matrix = text_matrix / text_matrix.norm(dim=-1, keepdim=True)


def clip_classify(rgb: np.ndarray) -> Dict:
    pil   = Image.fromarray(rgb)
    img_t = clip_preprocess(pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        img_feat = clip_model.encode_image(img_t)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        logits   = clip_model.logit_scale.exp() * (img_feat @ text_matrix.T)
        probs    = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    return {cls: float(p) for cls, p in zip(LABEL_NAMES, probs)}


print("CLIP zero-shot fallback ready. Text matrix:", text_matrix.shape)

---
## Stage 7 — RAG: NPUAP/EPUAP Guideline Retrieval Layer

24 curated chunks covering: staging definitions, prevention bundle,
documentation requirements, escalation triggers, and safety exceptions.

Embedded with `all-MiniLM-L6-v2` and indexed with FAISS.

In [ ]:
# =============================================================
# Stage 7a — Guideline corpus definition
# =============================================================
GUIDELINE_CHUNKS = [
    # Definitions
    {"id":"def_pi",    "cat":"definition",
     "text":"A pressure injury is localized damage to the skin and underlying soft tissue, usually over a bony prominence or related to a medical device. It occurs as a result of intense or prolonged pressure, or pressure in combination with shear."},
    # Staging
    {"id":"def_stage1","cat":"staging",
     "text":"Stage 1 Pressure Injury: Non-blanchable erythema of intact skin. Dark skin tones may not show visible blanching; colour may differ from surrounding area. Area may be painful, firm, soft, warmer or cooler."},
    {"id":"def_stage2","cat":"staging",
     "text":"Stage 2 Pressure Injury: Partial-thickness skin loss with exposed dermis. Wound bed is viable, pink or red, moist. May present as an intact or ruptured serum-filled blister."},
    {"id":"def_stage3","cat":"staging",
     "text":"Stage 3 Pressure Injury: Full-thickness skin loss. Adipose tissue visible; granulation tissue and epibole often present. Slough and/or eschar may be visible. Depth varies by anatomical location."},
    {"id":"def_stage4","cat":"staging",
     "text":"Stage 4 Pressure Injury: Full-thickness skin and tissue loss. Fascia, muscle, tendon, ligament, cartilage or bone exposed or directly palpable. Slough and/or eschar may be visible."},
    {"id":"def_unstage","cat":"staging",
     "text":"Unstageable Pressure Injury: Full-thickness skin and tissue loss where depth cannot be confirmed because it is obscured by slough or eschar. Remove slough/eschar to determine true depth and stage."},
    {"id":"def_dtpi",  "cat":"staging",
     "text":"Deep Tissue Pressure Injury (DTPI): Persistent non-blanchable deep red, maroon, or purple discoloration on intact skin, or blood-filled blister. Pain and firmness may precede colour change."},
    {"id":"def_device","cat":"definition",
     "text":"Medical Device-Related Pressure Injury: Caused by devices used for diagnostic or therapeutic purposes. Injury is confined to the area under the device and conforms to the pattern or shape of the device."},
    # Prevention
    {"id":"prev_repos",    "cat":"prevention",
     "text":"Reposition individuals at risk at least every 2 hours on a standard mattress, or every 4 hours on a pressure-redistribution surface. Document repositioning frequency and compliance."},
    {"id":"prev_surface",  "cat":"prevention",
     "text":"Use a pressure-redistribution support surface for all patients at elevated risk. Evaluate the surface regularly for bottoming out. Reactive surfaces reduce pressure passively; active surfaces alternate pressure cyclically."},
    {"id":"prev_moisture", "cat":"prevention",
     "text":"Manage skin moisture: cleanse skin promptly after incontinence. Apply moisture barrier or skin protectant. Prolonged moisture exposure increases friction and shear risk."},
    {"id":"prev_nutrition","cat":"prevention",
     "text":"Screen nutritional status for all at-risk individuals using a validated tool. Provide adequate calories, protein, and micronutrients. Consult dietitian for patients with nutritional deficits."},
    {"id":"prev_skin",     "cat":"prevention",
     "text":"Conduct a head-to-toe skin assessment at least daily for at-risk patients. Focus on bony prominences: sacrum, coccyx, heels, trochanters, ischial tuberosities, occiput."},
    {"id":"prev_heel",     "cat":"prevention",
     "text":"Elevate heels off the bed surface using a heel-suspension device. Inspect heels at least twice daily. Avoid donut-shaped devices which create focal pressure around the heel."},
    {"id":"prev_friction", "cat":"prevention",
     "text":"Reduce friction and shear: use lifting equipment, slide sheets, or draw sheets when moving patients. Keep head-of-bed elevation at or below 30 degrees to reduce sacral shear."},
    # Documentation
    {"id":"doc_assess",   "cat":"documentation",
     "text":"Document wound assessment: location, size (length x width x depth in cm), stage/classification, wound bed description (tissue type percentages), exudate amount and character, periwound skin condition, pain level."},
    {"id":"doc_photo",    "cat":"documentation",
     "text":"Photograph pressure injuries at baseline and per facility policy schedule. Use consistent lighting, angle, and distance. Include a measurement reference (ruler or colour patch) if possible."},
    {"id":"doc_progress", "cat":"documentation",
     "text":"Reassess pressure injury status at each dressing change or at least weekly. Document wound trajectory: improving, stable, or deteriorating. Update the care plan based on healing progress."},
    # Escalation
    {"id":"esc_clinician","cat":"escalation",
     "text":"Escalate to wound care specialist or medical provider if: wound deteriorates or does not improve in 2 weeks, signs of infection (erythema, warmth, purulent exudate, odour, fever), or staging is uncertain."},
    # Safety
    {"id":"esc_darkskin",   "cat":"safety",
     "text":"In individuals with dark skin tones, Stage 1 pressure injuries may not show visible blanching or erythema. Assess for changes in skin temperature, firmness, or pain, which may precede visible colour changes."},
    {"id":"esc_heel_eschar","cat":"safety",
     "text":"Heel pressure injuries with stable, dry, adherent eschar in patients without oedema, erythema, fluctuance, or drainage should NOT be unroofed. The eschar serves as the body's natural cover."},
    # Uncertainty
    {"id":"uncertain","cat":"uncertainty",
     "text":"If wound appearance does not clearly match a defined stage, or if image quality is poor, document as unable to stage at this time and arrange in-person wound care evaluation. Do not guess stage from an unclear image."},
    # Measurement
    {"id":"meas_method","cat":"measurement",
     "text":"Measure wound dimensions consistently: length (head-to-toe axis), width (perpendicular), depth (deepest point). Report in centimetres. Use the clock method for undermining and tunnelling documentation."},
    {"id":"meas_healing","cat":"measurement",
     "text":"Track wound area over time as a key healing indicator. A reduction of 20-40% in wound area within 4 weeks is associated with complete healing. Lack of progress warrants care plan reassessment."},
]

print(f"Guideline corpus: {len(GUIDELINE_CHUNKS)} chunks")
print(pd.Series([c['cat'] for c in GUIDELINE_CHUNKS]).value_counts().to_dict())

In [ ]:
# =============================================================
# Stage 7b — Build and save FAISS index
# =============================================================
rag_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

chunk_texts = [c["text"] for c in GUIDELINE_CHUNKS]
embeddings  = rag_encoder.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)
embeddings  = (embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)).astype(np.float32)

DIM         = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(DIM)
faiss_index.add(embeddings)

FAISS_PATH  = RAG_DIR / "guideline_index.faiss"
CHUNKS_PATH = RAG_DIR / "guideline_chunks.json"
faiss.write_index(faiss_index, str(FAISS_PATH))
with open(CHUNKS_PATH, "w") as f:
    json.dump(GUIDELINE_CHUNKS, f, indent=2)

print(f"FAISS index: {faiss_index.ntotal} vectors, dim={DIM}")


def retrieve_guidelines(query: str, top_k: int = 4) -> List[Dict]:
    q = rag_encoder.encode([query], convert_to_numpy=True)
    q = (q / np.linalg.norm(q, axis=1, keepdims=True)).astype(np.float32)
    scores, indices = faiss_index.search(q, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        c = dict(GUIDELINE_CHUNKS[idx])
        c["retrieval_score"] = round(float(score), 4)
        results.append(c)
    return results


# Smoke test
test_ret = retrieve_guidelines("reposition turning schedule pressure prevention")
print("\nTest retrieval:")
for r in test_ret:
    print(f"  [{r['cat']}] {r['id']}  score={r['retrieval_score']}")

---
## Stage 8 — Full Inference Pipeline

```
Image
  └─▶ GrabCut ROI + quality check
       └─▶ Contour / tissue colour / LBP features
            └─▶ MobileNetV3 concern score
                 ├── confident  ──▶ use MobileNet result
                 └── abstained  ──▶ CLIP blend fallback
                                        └─▶ FAISS RAG (top-4 chunks)
                                                 └─▶ JSON structured note
```

In [ ]:
# =============================================================
# Stage 8 — Full inference pipeline
# =============================================================
_infer_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])


def mobilenet_score(rgb: np.ndarray) -> Dict:
    model.eval()
    img_t = _infer_tf(Image.fromarray(rgb)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(model(img_t), dim=1).cpu().numpy()[0]
    top = int(np.argmax(probs))
    return {
        "top_class":            LABEL_NAMES[top],
        "top_prob":             float(probs[top]),
        "class_probabilities":  {c: float(p) for c, p in zip(LABEL_NAMES, probs)},
        "pressure_injury_prob": float(probs[0]),
        "abstained":            float(probs[top]) < ABSTAIN_THRESHOLD,
    }


def merge_concern(mn: Dict, clip: Optional[Dict]) -> Dict:
    if mn["abstained"] and clip:
        pi_prob = 0.4 * mn["pressure_injury_prob"] + 0.6 * clip.get("pressure_injury", 0.0)
        method  = "clip_fallback_blend"
    else:
        pi_prob = mn["pressure_injury_prob"]
        method  = "mobilenet"

    if   pi_prob >= 0.60: stage = {"label": "pressure-injury-like lesion",  "confidence": "medium"}
    elif pi_prob >= 0.35: stage = {"label": "possible pressure-injury-like", "confidence": "low"}
    else:                 stage = {"label": "unable_to_determine",           "confidence": "low"}

    stage["note"]   = "Demo output only. Clinician review required."
    stage["method"] = method
    return {"pressure_injury_probability": round(pi_prob, 4),
            "stage_suspicion": stage, "mobilenet": mn, "clip": clip}


def prevention_checklist(risk: Dict, pi_prob: float) -> List[str]:
    items = []
    if risk.get("mobility_limited") or pi_prob >= 0.35:
        items += ["Reposition/offloading schedule review",
                  "Document turning/repositioning compliance"]
    if risk.get("moisture_issue"):
        items += ["Moisture/microclimate management",
                  "Barrier protection and frequent skin inspection"]
    if risk.get("nutrition_risk"):  items.append("Nutrition/hydration assessment — consult dietitian")
    if risk.get("device_present"): items.append("Medical device pressure-point check (twice daily)")
    items.append("Confirm support surface appropriateness" if risk.get("support_surface_in_use")
                 else "Evaluate need for pressure-redistribution support surface")
    if risk.get("previous_pressure_injury"): items.append("Heightened vigilance: prior PI history")
    items.append("Escalate to wound-care review if lesion worsens or uncertainty remains")
    return items


def full_inference(image_path: str, risk_form: Dict,
                   use_clip: bool = True, top_k: int = 4) -> Dict:
    # Load
    bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if bgr is None:
        return {"error": f"Cannot load: {image_path}"}
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # ROI
    quality  = compute_quality_flags(rgb)
    mask     = largest_component(grabcut_roi(rgb))
    ys, xs   = np.where(mask > 0)
    bbox     = [int(xs.min()),int(ys.min()),int(xs.max()),int(ys.max())] if len(xs)>0 else None
    contour  = contour_metrics(mask)
    colours  = tissue_colour_ratios(rgb, mask)

    # Crop
    if bbox:
        x1,y1,x2,y2 = bbox; h,w = rgb.shape[:2]; p=10
        crop = rgb[max(0,y1-p):min(h,y2+p+1), max(0,x1-p):min(w,x2+p+1)]
    else:
        crop = rgb

    # Classify
    mn     = mobilenet_score(crop)
    clip_r = clip_classify(crop) if (use_clip and mn["abstained"]) else None
    concern = merge_concern(mn, clip_r)
    pi_p    = concern["pressure_injury_probability"]

    # RAG
    rag_chunks = retrieve_guidelines(
        f"pressure injury {concern['stage_suspicion']['label']} prevention documentation", top_k
    )

    # Checklist
    checklist = prevention_checklist(risk_form, pi_p)

    return {
        "schema_version": "wounddoc_lite_v1",
        "meta": {
            "image_path": image_path,
            "disclaimer": "Demo only. NOT clinically validated. Clinician review required.",
        },
        "image_quality": {"flags": quality, "usable": len(quality) < 2},
        "roi": {
            "found": bbox is not None, "bbox": bbox,
            "area_px": contour["area_px"], "area_mm2": contour["area_mm2"],
            "perimeter_mm": contour["perimeter_mm"], "circularity": contour["circularity"],
            "tissue_colours": colours,
        },
        "concern": concern,
        "risk_form": risk_form,
        "prevention_checklist": checklist,
        "retrieved_guidance": [
            {"id": c["id"], "cat": c["cat"],
             "score": c["retrieval_score"], "excerpt": c["text"][:200] + "..."}
            for c in rag_chunks
        ],
        "structured_note": {
            "wound_concern":      concern["stage_suspicion"]["label"],
            "confidence":         concern["stage_suspicion"]["confidence"],
            "pi_probability":     pi_p,
            "roi_area_mm2":       contour["area_mm2"],
            "tissue_composition": colours,
            "quality_flags":      quality,
            "risk_factors":       [k for k,v in risk_form.items() if v is True],
            "prevention_actions": checklist,
            "narrative": (
                f"Demo AI review: wound concern='{concern['stage_suspicion']['label']}' "
                f"(PI probability={pi_p:.3f}, confidence={concern['stage_suspicion']['confidence']}). "
                f"ROI {'detected' if bbox else 'not detected'}. "
                f"Quality flags: {quality if quality else 'none'}. "
                f"CLINICIAN REVIEW REQUIRED BEFORE ANY CLINICAL ACTION."
            ),
            "clinician_review_state": "pending",
        },
    }


print("Inference pipeline ready.")
print(f"  MobileNetV3-Small (abstain={ABSTAIN_THRESHOLD})")
print(f"  CLIP ViT-B/32 fallback")
print(f"  FAISS RAG ({faiss_index.ntotal} chunks)")

---
## Stage 9 — Smoke Test on Sample Images

In [ ]:
# =============================================================
# Stage 9 — Smoke test: 3 random test images end-to-end
# =============================================================
DEMO_RISK = {
    "mobility_limited": True,  "moisture_issue": True,
    "nutrition_risk":   False, "device_present": False,
    "previous_pressure_injury": False, "support_surface_in_use": True,
}

sample_paths  = test_df.sample(min(3, len(test_df)), random_state=SEED)["image_path"].tolist()
smoke_outputs = []

for img_path in sample_paths:
    result = full_inference(img_path, DEMO_RISK)
    smoke_outputs.append(result)

    # Visualise
    bgr = cv2.imread(img_path)
    if bgr is not None:
        rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        mask = largest_component(grabcut_roi(rgb))
        ov   = rgb.copy()
        ov[mask > 0] = (0.65*ov[mask>0] + 0.35*np.array([255,50,50])).astype(np.uint8)

        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].imshow(rgb); axes[0].set_title("Original");   axes[0].axis("off")
        axes[1].imshow(ov);  axes[1].set_title("ROI Overlay"); axes[1].axis("off")

        concern = result.get("concern", {})
        pi_p    = concern.get("pressure_injury_probability", 0)
        label   = concern.get("stage_suspicion", {}).get("label", "n/a")
        fig.suptitle(f"PI prob={pi_p:.3f} | {label}", fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.show(); plt.close()

    compact = {
        "image":             os.path.basename(img_path),
        "pi_prob":           result.get("concern",{}).get("pressure_injury_probability"),
        "stage_suspicion":   result.get("concern",{}).get("stage_suspicion",{}).get("label"),
        "quality_flags":     result.get("image_quality",{}).get("flags"),
        "roi_area_mm2":      result.get("roi",{}).get("area_mm2"),
        "tissue_colours":    result.get("roi",{}).get("tissue_colours"),
        "prevention":        result.get("prevention_checklist"),
    }
    print(json.dumps(compact, indent=2))
    print("="*80)

---
## Stage 10 — Save & Package All Artefacts

In [ ]:
# =============================================================
# Stage 10 — Persist evaluation summary + smoke outputs
# =============================================================
import datetime

eval_summary = {
    "timestamp":          datetime.datetime.now().isoformat(),
    "model":              "mobilenet_v3_small",
    "test_accuracy":      float(test_acc),
    "abstain_threshold":  ABSTAIN_THRESHOLD,
    "abstain_rate":       float(abstain_rate),
    "confident_accuracy": float(conf_acc),
    "pressure_injury_auroc": float(auroc) if auroc is not None else None,
    "label_names":        LABEL_NAMES,
    "dataset_sizes":      {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
}

with open(OUTPUT_DIR / "eval_summary.json", "w") as f:
    json.dump(eval_summary, f, indent=2)

with open(OUTPUT_DIR / "smoke_test_outputs.json", "w") as f:
    json.dump(smoke_outputs, f, indent=2)

print("=== Artefact inventory ===")
for p in sorted(BASE_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(BASE_DIR)}  ({p.stat().st_size/1024:.0f} KB)")

print("\n=== Eval summary ===")
print(json.dumps(eval_summary, indent=2))

In [ ]:
# =============================================================
# Stage 10b — Package for download (model + RAG + eval)
# =============================================================
PACKAGE_ZIP = Path("/content/wounddoc_artefacts.zip")

pack_files = [
    MODEL_PATH,
    FAISS_PATH,
    CHUNKS_PATH,
    OUTPUT_DIR / "eval_summary.json",
    OUTPUT_DIR / "training_curves.png",
    OUTPUT_DIR / "confusion_matrix.png",
    OUTPUT_DIR / "confidence_distribution.png",
]

with zipfile.ZipFile(PACKAGE_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in pack_files:
        if Path(p).exists():
            zf.write(p, arcname=Path(p).name)

print(f"Packaged → {PACKAGE_ZIP}  ({PACKAGE_ZIP.stat().st_size/1024:.0f} KB)")

try:
    from google.colab import files
    files.download(str(PACKAGE_ZIP))
    print("Download triggered.")
except Exception:
    print("Not in Colab — artefacts at", PACKAGE_ZIP)

---
## Summary — Artefacts & Architecture

| Artefact | Path | Used for |
|---|---|---|
| `wound_concern_mobilenet.pt` | `models/` | Main concern classifier |
| `guideline_index.faiss` | `rag/` | Fast guideline retrieval |
| `guideline_chunks.json` | `rag/` | Guideline text source |
| `manifest.csv` | `unified/` | Full image manifest |
| `features.jsonl` | `unified/` | Pre-computed wound features |
| `eval_summary.json` | `outputs/` | Training/test metrics |

### Model choices (accessibility rationale)

| Component | Model | Size | Rationale |
|---|---|---|---|
| Concern classifier | MobileNetV3-Small | ~2.5 MB | Edge-deployable, runs on CPU |
| Zero-shot fallback | CLIP ViT-B/32 | ~340 MB | No domain labels needed |
| RAG encoder | all-MiniLM-L6-v2 | ~22 MB | Tiny, fast, accurate semantic sim |
| RAG index | FAISS FlatIP | <1 MB | In-memory, no server needed |

### Wound measurement coverage
The pipeline reports:
- Contour area (px + estimated mm²)
- Perimeter / circumference (mm)
- Bounding box (length × width proxy)
- Circularity index
- Tissue colour composition (granulation / slough / eschar / epithelialising)
- Image quality flags
- Healing trajectory (track area_mm2 across visits)

---
> **Hackathon demo only. All outputs require clinician review before any clinical action.**
